In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                           roc_auc_score, confusion_matrix, classification_report,
                           roc_curve, auc, precision_recall_curve)
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")

# ============================================
# STEP 1: DATA LOADING
# ============================================

def load_data(train_path, test_path, aaindex_path):
    """
    Load train, test and AAIndex data
    """
    print("="*50)
    print("STEP 1: LOADING DATA")
    print("="*50)
    
    # Load train and test data
    train_df = pd.read_csv("../data/features_pca/train_dataset.csv")
    test_df = pd.read_csv("../data/features_pca/test_dataset.csv")
    
    print(f"Train data shape: {train_df.shape}")
    print(f"Test data shape: {test_df.shape}")
    print(f"Train columns: {train_df.columns.tolist()}")
    print(f"Test columns: {test_df.columns.tolist()}")
    
    # Load AAIndex data (assuming first column is amino acid)
    aaindex_df = pd.read_csv("../data/features_pca/AAindex_20x566_matrix.csv")
    
    # Set first column as index (amino acids)
    if aaindex_df.columns[0] == 'Unnamed: 0' or aaindex_df.columns[0] == '':
        # If first column doesn't have a name, assume it's the amino acid column
        aaindex_df = aaindex_df.rename(columns={aaindex_df.columns[0]: 'Amino_Acid'})
    else:
        aaindex_df = aaindex_df.rename(columns={aaindex_df.columns[0]: 'Amino_Acid'})
    
    aaindex_df.set_index('Amino_Acid', inplace=True)
    
    print(f"\nAAIndex data shape: {aaindex_df.shape}")
    print(f"Number of physicochemical features: {len(aaindex_df.columns)}")
    print(f"Amino acids available: {aaindex_df.index.tolist()}")
    
    return train_df, test_df, aaindex_df

# ============================================
# STEP 2: FEATURE ENGINEERING
# ============================================

def extract_sequence_features(window_seq, aaindex_df):
    """
    Extract physicochemical features from a sequence window
    """
    features = {}
    
    for aa in window_seq:
        if aa in aaindex_df.index:
            aa_features = aaindex_df.loc[aa].to_dict()
            for feature_name, value in aa_features.items():
                if feature_name not in features:
                    features[feature_name] = []
                features[feature_name].append(value)
        else:
            # If amino acid not found, use mean values
            for feature_name in aaindex_df.columns:
                if feature_name not in features:
                    features[feature_name] = []
                features[feature_name].append(aaindex_df[feature_name].mean())
    
    # Calculate statistics for each feature
    final_features = {}
    for feature_name, values in features.items():
        final_features[f'{feature_name}_mean'] = np.mean(values)
        final_features[f'{feature_name}_std'] = np.std(values)
        final_features[f'{feature_name}_min'] = np.min(values)
        final_features[f'{feature_name}_max'] = np.max(values)
        final_features[f'{feature_name}_range'] = np.max(values) - np.min(values)
    
    return final_features

def prepare_features(df, aaindex_df):
    """
    Prepare features for all sequences
    """
    print("\n" + "="*50)
    print("STEP 2: FEATURE ENGINEERING")
    print("="*50)
    
    features_list = []
    labels = []
    protein_ids = []
    positions = []
    
    for idx, row in df.iterrows():
        window_seq = row['window']
        label = row['label']
        
        # Extract features for this window
        window_features = extract_sequence_features(window_seq, aaindex_df)
        features_list.append(window_features)
        labels.append(label)
        protein_ids.append(row['protein_id'])
        positions.append(row['position'])
        
        if idx % 1000 == 0 and idx > 0:
            print(f"Processed {idx} sequences...")
    
    # Convert to DataFrame
    features_df = pd.DataFrame(features_list)
    features_df['protein_id'] = protein_ids
    features_df['position'] = positions
    features_df['label'] = labels
    
    print(f"\nTotal sequences processed: {len(features_df)}")
    print(f"Total features created: {len(features_df.columns) - 3}")  # -3 for protein_id, position, label
    print(f"Class distribution:")
    print(features_df['label'].value_counts())
    
    return features_df

# ============================================
# STEP 3: PCA ANALYSIS
# ============================================

def perform_pca_analysis(X_scaled, y, n_components=None):
    """
    Perform PCA analysis and visualization
    """
    print("\n" + "="*50)
    print("STEP 3: PCA ANALYSIS")
    print("="*50)
    
    # Perform PCA
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)
    
    # Calculate explained variance
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_variance_ratio = np.cumsum(explained_variance_ratio)
    
    # Find number of components for 95% variance
    n_components_95 = np.argmax(cumulative_variance_ratio >= 0.95) + 1
    
    print(f"Total components: {len(explained_variance_ratio)}")
    print(f"Components for 95% variance: {n_components_95}")
    print(f"Variance explained by top 10 components: {cumulative_variance_ratio[:10]}")
    print(f"Total variance explained: {cumulative_variance_ratio[-1]:.2%}")
    
    # Create PCA visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Plot 1: Scree plot
    axes[0,0].plot(range(1, min(21, len(explained_variance_ratio)+1)), 
                   explained_variance_ratio[:20], 'bo-')
    axes[0,0].set_xlabel('Principal Component')
    axes[0,0].set_ylabel('Explained Variance Ratio')
    axes[0,0].set_title('Scree Plot (First 20 Components)')
    axes[0,0].grid(True, alpha=0.3)
    
    # Plot 2: Cumulative variance
    axes[0,1].plot(range(1, len(cumulative_variance_ratio)+1), 
                   cumulative_variance_ratio, 'ro-')
    axes[0,1].axhline(y=0.95, color='g', linestyle='--', label='95% threshold')
    axes[0,1].axvline(x=n_components_95, color='b', linestyle='--', 
                      label=f'{n_components_95} components')
    axes[0,1].set_xlabel('Number of Components')
    axes[0,1].set_ylabel('Cumulative Explained Variance')
    axes[0,1].set_title('Cumulative Explained Variance')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # Plot 3: First two components scatter (colored by labels)
    scatter = axes[0,2].scatter(X_pca[:, 0], X_pca[:, 1], 
                               c=y, cmap='coolwarm', alpha=0.6, s=10)
    axes[0,2].set_xlabel(f'PC1 ({explained_variance_ratio[0]:.2%})')
    axes[0,2].set_ylabel(f'PC2 ({explained_variance_ratio[1]:.2%})')
    axes[0,2].set_title('PCA: First Two Components')
    plt.colorbar(scatter, ax=axes[0,2])
    
    # Plot 4: Variance distribution pie chart
    variance_pcs = explained_variance_ratio[:5]
    other_variance = 1 - sum(variance_pcs)
    labels_pie = [f'PC{i+1}' for i in range(5)] + ['Others']
    sizes = list(variance_pcs) + [other_variance]
    axes[1,0].pie(sizes, labels=labels_pie, autopct='%1.1f%%', startangle=90)
    axes[1,0].set_title('Variance Distribution (Top 5 PCs)')
    
    # Plot 5: Component loadings heatmap (top 10 PCs, top 20 features)
    n_pcs_show = min(10, pca.n_components_)
    n_features_show = min(20, X_scaled.shape[1])
    
    loadings = pd.DataFrame(
        pca.components_[:n_pcs_show, :n_features_show],
        columns=X_scaled.columns[:n_features_show],
        index=[f'PC{i+1}' for i in range(n_pcs_show)]
    )
    sns.heatmap(loadings, annot=False, cmap='RdBu_r', center=0,
                ax=axes[1,1], cbar_kws={'label': 'Loading'}, 
                xticklabels=False)
    axes[1,1].set_title(f'PCA Loadings (First {n_pcs_show} PCs × First {n_features_show} Features)')
    
    # Plot 6: PC3 vs PC4
    if X_pca.shape[1] >= 4:
        scatter2 = axes[1,2].scatter(X_pca[:, 2], X_pca[:, 3], 
                                    c=y, cmap='coolwarm', alpha=0.6, s=10)
        axes[1,2].set_xlabel(f'PC3 ({explained_variance_ratio[2]:.2%})')
        axes[1,2].set_ylabel(f'PC4 ({explained_variance_ratio[3]:.2%})')
        axes[1,2].set_title('PCA: PC3 vs PC4')
        plt.colorbar(scatter2, ax=axes[1,2])
    
    plt.tight_layout()
    plt.savefig('pca_analysis_detailed.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Save PCA results
    pca_results = pd.DataFrame(
        X_pca, 
        columns=[f'PC{i+1}' for i in range(X_pca.shape[1])]
    )
    pca_results['label'] = y.values
    
    print(f"\nPCA completed. Shape of transformed data: {X_pca.shape}")
    
    return pca, X_pca, n_components_95

# ============================================
# STEP 4: RANDOM FOREST MODEL WITH SPECIFIED PARAMETERS
# ============================================

def train_random_forest_with_params(X_train, y_train, X_test, y_test):
    """
    Train Random Forest classifier with specified parameters
    """
    print("\n" + "="*50)
    print("STEP 4: RANDOM FOREST TRAINING WITH SPECIFIED PARAMETERS")
    print("="*50)
    
    # Initialize Random Forest with specified parameters
    rf_model = RandomForestClassifier(
        n_estimators=500,
        max_depth=40,
        max_features="sqrt",
        min_samples_leaf=2,
        min_samples_split=5,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42
    )
    
    print("Random Forest Parameters:")
    print(f"  n_estimators: 500")
    print(f"  max_depth: 40")
    print(f"  max_features: sqrt")
    print(f"  min_samples_leaf: 2")
    print(f"  min_samples_split: 5")
    print(f"  class_weight: balanced_subsample")
    print(f"  random_state: 42")
    print(f"  n_jobs: -1 (using all cores)")
    
    # Train the model
    print("\nTraining model...")
    rf_model.fit(X_train, y_train)
    
    # Cross-validation scores
    print("\nPerforming 5-fold cross-validation...")
    cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, 
                               scoring='roc_auc', n_jobs=-1)
    print(f"Cross-validation ROC-AUC scores: {cv_scores}")
    print(f"Mean CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
    
    # Make predictions
    y_pred = rf_model.predict(X_test)
    y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
    
    return rf_model, y_pred, y_pred_proba, cv_scores

# ============================================
# STEP 5: MODEL EVALUATION
# ============================================

def evaluate_model(y_test, y_pred, y_pred_proba, model, feature_names, cv_scores):
    """
    Comprehensive model evaluation
    """
    print("\n" + "="*50)
    print("STEP 5: MODEL EVALUATION")
    print("="*50)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f"\n{'='*30}")
    print("MODEL PERFORMANCE METRICS")
    print(f"{'='*30}")
    print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
    print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    print(f"\nConfusion Matrix:")
    print(f"              Predicted")
    print(f"              Negative  Positive")
    print(f"Actual Negative   {cm[0,0]:4d}      {cm[0,1]:4d}")
    print(f"       Positive   {cm[1,0]:4d}      {cm[1,1]:4d}")
    
    # Calculate additional metrics
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp)
    npv = tn / (tn + fn)  # Negative Predictive Value
    mcc = (tp*tn - fp*fn) / np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))  # Matthews Correlation Coefficient
    
    print(f"\nAdditional Metrics:")
    print(f"Specificity: {specificity:.4f} ({specificity*100:.2f}%)")
    print(f"NPV: {npv:.4f} ({npv*100:.2f}%)")
    print(f"MCC: {mcc:.4f}")
    
    # Classification Report
    print(f"\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))
    
    metrics_dict = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'f1': f1,
        'roc_auc': roc_auc,
        'mcc': mcc,
        'npv': npv,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }
    
    return metrics_dict

# ============================================
# STEP 6: FEATURE IMPORTANCE ANALYSIS
# ============================================

def analyze_feature_importance(model, feature_names, top_n=30):
    """
    Analyze and visualize feature importance
    """
    print("\n" + "="*50)
    print("STEP 6: FEATURE IMPORTANCE ANALYSIS")
    print("="*50)
    
    feature_importance = model.feature_importances_
    
    # Create importance DataFrame
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    # Calculate cumulative importance
    importance_df['cumulative_importance'] = importance_df['importance'].cumsum()
    
    print(f"\nTop {top_n} Most Important Features:")
    print(importance_df.head(top_n).to_string(index=False))
    
    # Find number of features for 95% importance
    n_features_95 = np.argmax(importance_df['cumulative_importance'] >= 0.95) + 1
    print(f"\nFeatures needed for 95% cumulative importance: {n_features_95}")
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Plot 1: Top 20 feature importance
    top_20 = importance_df.head(20)
    axes[0,0].barh(range(len(top_20)), top_20['importance'].values)
    axes[0,0].set_yticks(range(len(top_20)))
    axes[0,0].set_yticklabels(top_20['feature'].values)
    axes[0,0].invert_yaxis()
    axes[0,0].set_xlabel('Importance')
    axes[0,0].set_title('Top 20 Feature Importances')
    
    # Plot 2: Cumulative importance
    axes[0,1].plot(range(1, len(importance_df)+1), 
                   importance_df['cumulative_importance'].values, 'b-')
    axes[0,1].axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
    axes[0,1].axvline(x=n_features_95, color='g', linestyle='--', 
                      label=f'{n_features_95} features')
    axes[0,1].set_xlabel('Number of Features')
    axes[0,1].set_ylabel('Cumulative Importance')
    axes[0,1].set_title('Cumulative Feature Importance')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # Plot 3: Importance distribution
    axes[1,0].hist(feature_importance, bins=50, edgecolor='black')
    axes[1,0].set_xlabel('Importance Value')
    axes[1,0].set_ylabel('Frequency')
    axes[1,0].set_title('Distribution of Feature Importances')
    
    # Plot 4: Top 10 features with values
    top_10 = importance_df.head(10)
    axes[1,1].bar(range(len(top_10)), top_10['importance'].values)
    axes[1,1].set_xticks(range(len(top_10)))
    axes[1,1].set_xticklabels(top_10['feature'].values, rotation=45, ha='right')
    axes[1,1].set_ylabel('Importance')
    axes[1,1].set_title('Top 10 Feature Importances')
    
    plt.tight_layout()
    plt.savefig('feature_importance_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return importance_df

# ============================================
# STEP 7: COMPREHENSIVE VISUALIZATIONS
# ============================================

def create_comprehensive_visualizations(y_test, y_pred, y_pred_proba, metrics_dict):
    """
    Create all evaluation visualization plots
    """
    print("\n" + "="*50)
    print("STEP 7: CREATING COMPREHENSIVE VISUALIZATIONS")
    print("="*50)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    axes[0,0].plot(fpr, tpr, color='darkorange', lw=2, 
                   label=f'ROC curve (AUC = {roc_auc:.3f})')
    axes[0,0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[0,0].fill_between(fpr, tpr, alpha=0.2, color='darkorange')
    axes[0,0].set_xlim([0.0, 1.0])
    axes[0,0].set_ylim([0.0, 1.05])
    axes[0,0].set_xlabel('False Positive Rate (1 - Specificity)')
    axes[0,0].set_ylabel('True Positive Rate (Sensitivity)')
    axes[0,0].set_title(f'ROC Curve (AUC = {roc_auc:.3f})')
    axes[0,0].legend(loc="lower right")
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. Confusion Matrix Heatmap
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,1],
                xticklabels=['Predicted 0', 'Predicted 1'],
                yticklabels=['Actual 0', 'Actual 1'])
    axes[0,1].set_xlabel('Predicted')
    axes[0,1].set_ylabel('Actual')
    axes[0,1].set_title('Confusion Matrix')
    
    # 3. Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    pr_auc = auc(recall, precision)
    
    axes[0,2].plot(recall, precision, color='green', lw=2,
                   label=f'PR curve (AUC = {pr_auc:.3f})')
    axes[0,2].fill_between(recall, precision, alpha=0.2, color='green')
    axes[0,2].set_xlabel('Recall')
    axes[0,2].set_ylabel('Precision')
    axes[0,2].set_title(f'Precision-Recall Curve (AUC = {pr_auc:.3f})')
    axes[0,2].legend(loc="lower left")
    axes[0,2].grid(True, alpha=0.3)
    
    # 4. Prediction Distribution
    axes[1,0].hist(y_pred_proba[y_test==0], bins=30, alpha=0.7, 
                   label='Class 0 (Negative)', color='blue', edgecolor='black')
    axes[1,0].hist(y_pred_proba[y_test==1], bins=30, alpha=0.7, 
                   label='Class 1 (Positive)', color='red', edgecolor='black')
    axes[1,0].axvline(x=0.5, color='black', linestyle='--', label='Decision boundary')
    axes[1,0].set_xlabel('Predicted Probability')
    axes[1,0].set_ylabel('Frequency')
    axes[1,0].set_title('Distribution of Predicted Probabilities')
    axes[1,0].legend()
    
    # 5. Metrics Bar Plot
    metrics_display = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1-Score', 'ROC-AUC']
    values = [metrics_dict['accuracy'], 
              metrics_dict['precision'],
              metrics_dict['recall'],
              metrics_dict['specificity'],
              metrics_dict['f1'],
              metrics_dict['roc_auc']]
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
    bars = axes[1,1].bar(metrics_display, values, color=colors, edgecolor='black')
    axes[1,1].set_ylim([0, 1])
    axes[1,1].set_ylabel('Score')
    axes[1,1].set_title('Model Performance Metrics')
    axes[1,1].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, v in zip(bars, values):
        axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                      f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    
    # 6. Cross-validation scores
    cv_scores = metrics_dict.get('cv_scores', [metrics_dict['cv_mean']]*5)
    axes[1,2].boxplot([cv_scores], widths=0.5)
    axes[1,2].scatter([1] * len(cv_scores), cv_scores, alpha=0.6, color='red')
    axes[1,2].set_xticklabels(['5-Fold CV'])
    axes[1,2].set_ylabel('ROC-AUC Score')
    axes[1,2].set_title(f'Cross-Validation Scores\nMean: {metrics_dict["cv_mean"]:.3f} (±{metrics_dict["cv_std"]:.3f})')
    axes[1,2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('comprehensive_evaluation.png', dpi=300, bbox_inches='tight')
    plt.show()

# ============================================
# STEP 8: COMPARE WITH/WITHOUT PCA
# ============================================

def compare_pca_impact(X_train_scaled, X_test_scaled, y_train, y_test, 
                       X_train_pca, X_test_pca, feature_names):
    """
    Compare model performance with and without PCA
    """
    print("\n" + "="*50)
    print("STEP 8: COMPARING MODEL WITH/WITHOUT PCA")
    print("="*50)
    
    # Train model without PCA
    print("\nTraining model WITHOUT PCA...")
    rf_no_pca = RandomForestClassifier(
        n_estimators=500,
        max_depth=40,
        max_features="sqrt",
        min_samples_leaf=2,
        min_samples_split=5,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42
    )
    rf_no_pca.fit(X_train_scaled, y_train)
    y_pred_no_pca = rf_no_pca.predict(X_test_scaled)
    y_proba_no_pca = rf_no_pca.predict_proba(X_test_scaled)[:, 1]
    
    # Train model WITH PCA
    print("Training model WITH PCA...")
    rf_with_pca = RandomForestClassifier(
        n_estimators=500,
        max_depth=40,
        max_features="sqrt",
        min_samples_leaf=2,
        min_samples_split=5,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42
    )
    rf_with_pca.fit(X_train_pca, y_train)
    y_pred_with_pca = rf_with_pca.predict(X_test_pca)
    y_proba_with_pca = rf_with_pca.predict_proba(X_test_pca)[:, 1]
    
    # Calculate metrics
    metrics_no_pca = {
        'accuracy': accuracy_score(y_test, y_pred_no_pca),
        'precision': precision_score(y_test, y_pred_no_pca),
        'recall': recall_score(y_test, y_pred_no_pca),
        'f1': f1_score(y_test, y_pred_no_pca),
        'roc_auc': roc_auc_score(y_test, y_proba_no_pca)
    }
    
    metrics_with_pca = {
        'accuracy': accuracy_score(y_test, y_pred_with_pca),
        'precision': precision_score(y_test, y_pred_with_pca),
        'recall': recall_score(y_test, y_pred_with_pca),
        'f1': f1_score(y_test, y_pred_with_pca),
        'roc_auc': roc_auc_score(y_test, y_proba_with_pca)
    }
    
    # Create comparison DataFrame
    comparison_df = pd.DataFrame({
        'Metric': metrics_no_pca.keys(),
        'Without PCA': metrics_no_pca.values(),
        'With PCA': metrics_with_pca.values(),
        'Difference': [metrics_with_pca[m] - metrics_no_pca[m] for m in metrics_no_pca.keys()]
    })
    
    print("\nComparison Results:")
    print(comparison_df.to_string(index=False))
    
    # Visualize comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Bar plot comparison
    x = np.arange(len(metrics_no_pca))
    width = 0.35
    
    axes[0].bar(x - width/2, list(metrics_no_pca.values()), width, 
                label='Without PCA', color='skyblue', edgecolor='black')
    axes[0].bar(x + width/2, list(metrics_with_pca.values()), width,
                label='With PCA', color='lightcoral', edgecolor='black')
    axes[0].set_xlabel('Metrics')
    axes[0].set_ylabel('Score')
    axes[0].set_title('Performance: With vs Without PCA')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(list(metrics_no_pca.keys()))
    axes[0].legend()
    axes[0].set_ylim([0, 1])
    
    # Difference plot
    diff_values = comparison_df['Difference'].values
    colors = ['green' if d > 0 else 'red' for d in diff_values]
    axes[1].bar(x, diff_values, color=colors, edgecolor='black')
    axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[1].set_xlabel('Metrics')
    axes[1].set_ylabel('Difference (With PCA - Without PCA)')
    axes[1].set_title('Impact of PCA on Performance')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(list(metrics_no_pca.keys()))
    
    plt.tight_layout()
    plt.savefig('pca_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return comparison_df

# ============================================
# MAIN EXECUTION
# ============================================

if __name__ == "__main__":
    
    # File paths (update these with your actual paths)
    train_path = "train.csv"
    test_path = "test.csv"
    aaindex_path = "aaindex.csv"
    
    try:
        # Step 1: Load data
        train_df, test_df, aaindex_df = load_data(train_path, test_path, aaindex_path)
        
        # Step 2: Prepare features
        train_features = prepare_features(train_df, aaindex_df)
        test_features = prepare_features(test_df, aaindex_df)
        
        # Separate features and labels
        feature_columns = [col for col in train_features.columns 
                          if col not in ['protein_id', 'position', 'label']]
        
        X_train = train_features[feature_columns]
        y_train = train_features['label']
        X_test = test_features[feature_columns]
        y_test = test_features['label']
        
        print(f"\nTraining set shape: {X_train.shape}")
        print(f"Test set shape: {X_test.shape}")
        print(f"Number of features: {len(feature_columns)}")
        
        # Handle any missing values
        X_train = X_train.fillna(X_train.mean())
        X_test = X_test.fillna(X_train.mean())
        
        # Scale features
        print("\nScaling features...")
        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(
            scaler.fit_transform(X_train),
            columns=X_train.columns
        )
        X_test_scaled = pd.DataFrame(
            scaler.transform(X_test),
            columns=X_test.columns
        )
        
        # Step 3: PCA Analysis
        pca, X_train_pca, n_components_95 = perform_pca_analysis(X_train_scaled, y_train)
        
        # Transform test data with PCA
        X_test_pca = pca.transform(X_test_scaled)
        
        # Step 4: Train Random Forest with specified parameters
        model, y_pred, y_pred_proba, cv_scores = train_random_forest_with_params(
            X_train_scaled, y_train, X_test_scaled, y_test
        )
        
        # Step 5: Evaluate Model
        metrics = evaluate_model(y_test, y_pred, y_pred_proba, 
                                model, feature_columns, cv_scores)
        metrics['cv_scores'] = cv_scores
        
        # Step 6: Feature Importance Analysis
        importance_df = analyze_feature_importance(model, feature_columns)
        
        # Save feature importance to CSV
        importance_df.to_csv('feature_importance.csv', index=False)
        print(f"\nFeature importance saved to 'feature_importance.csv'")
        
        # Step 7: Create Comprehensive Visualizations
        create_comprehensive_visualizations(y_test, y_pred, y_pred_proba, metrics)
        
        # Step 8: Compare with PCA-transformed data
        # Train model on PCA data
        print("\n" + "="*50)
        print("Training model on PCA-transformed data...")
        print("="*50)
        
        model_pca, y_pred_pca, y_proba_pca, cv_scores_pca = train_random_forest_with_params(
            X_train_pca, y_train, X_test_pca, y_test
        )
        
        # Compare results
        comparison = compare_pca_impact(
            X_train_scaled, X_test_scaled, y_train, y_test,
            X_train_pca, X_test_pca, feature_columns
        )
        
        # Save predictions
        predictions_df = test_features[['protein_id', 'position', 'label']].copy()
        predictions_df['predicted_label'] = y_pred
        predictions_df['prediction_probability'] = y_pred_proba
        predictions_df.to_csv('predictions.csv', index=False)
        print(f"\nPredictions saved to 'predictions.csv'")
        
        # Save all metrics to file
        metrics_df = pd.DataFrame([metrics])
        metrics_df.to_csv('model_metrics.csv', index=False)
        print(f"Model metrics saved to 'model_metrics.csv'")
        
        # Final Summary
        print("\n" + "="*60)
        print("FINAL ANALYSIS SUMMARY")
        print("="*60)
        print(f"✓ Total features extracted: {len(feature_columns)}")
        print(f"✓ PCA components for 95% variance: {n_components_95}")
        print(f"✓ Random Forest model trained with specified parameters")
        print(f"\n✓ Model Performance (without PCA):")
        print(f"  - Accuracy:  {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)")
        print(f"  - ROC-AUC:   {metrics['roc_auc']:.4f}")
        print(f"  - F1-Score:  {metrics['f1']:.4f}")
        print(f"  - Precision: {metrics['precision']:.4f}")
        print(f"  - Recall:    {metrics['recall']:.4f}")
        print(f"  - MCC:       {metrics['mcc']:.4f}")
        
        print(f"\n✓ Files generated:")
        print(f"  - pca_analysis_detailed.png")
        print(f"  - feature_importance_analysis.png")
        print(f"  - comprehensive_evaluation.png")
        print(f"  - pca_comparison.png")
        print(f"  - feature_importance.csv")
        print(f"  - predictions.csv")
        print(f"  - model_metrics.csv")
        
        print(f"\n✓ Best features (top 5):")
        for idx, row in importance_df.head(5).iterrows():
            print(f"  {idx+1}. {row['feature']}: {row['importance']:.4f}")
        
    except FileNotFoundError as e:
        print(f"\n❌ Error: File not found. Please check the file paths.")
        print(f"   Details: {e}")
        print("\nPlease update the file paths in the main section:")
        print("   train_path = 'your_train.csv'")
        print("   test_path = 'your_test.csv'")
        print("   aaindex_path = 'your_aaindex.csv'")
        
    except Exception as e:
        print(f"\n❌ Error occurred: {e}")
        import traceback
        traceback.print_exc()

STEP 1: LOADING DATA
Train data shape: (55292, 4)
Test data shape: (17387, 4)
Train columns: ['protein_id', 'position', 'window', 'label']
Test columns: ['protein_id', 'position', 'window', 'label']

AAIndex data shape: (20, 553)
Number of physicochemical features: 553
Amino acids available: ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

STEP 2: FEATURE ENGINEERING
Processed 1000 sequences...
Processed 2000 sequences...
Processed 3000 sequences...
Processed 4000 sequences...
Processed 5000 sequences...
Processed 6000 sequences...


KeyboardInterrupt: 